In [ ]:
# Loading the data

import pandas as pd
import numpy as np

# Load returns matrix
df_returns = pd.read_csv("matrix_returns.csv")

# Load company names
with open("company_names.csv", "r") as f:
    company_names = [line.strip() for line in f]

# Remove the first entry if it is the header
company_names = company_names[1:]

# Verify
print(df_returns.shape)
print(company_names[:50])
print(len(company_names))

In [ ]:
# Plotting the time series of NVDA and AAPL

import matplotlib.pyplot as plt

# Same y-axis scale for both series
ymin = min(df_returns.iloc[0].min(), df_returns.iloc[1].min())
ymax = max(df_returns.iloc[0].max(), df_returns.iloc[1].max())

plt.style.use("default")

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 6), sharex=True, sharey=True, facecolor="white"
)

fig.patch.set_facecolor("white")
ax1.set_facecolor("white")
ax2.set_facecolor("white")

# NVDA
ax1.plot(df_returns.iloc[0].values, color="blue", lw=1)
ax1.set_title("NVDA", fontsize=18)
ax1.set_ylabel("Log-return", fontsize=16)
ax1.set_ylim(ymin, ymax)
ax1.grid(True, alpha=0.15)

# AAPL
ax2.plot(df_returns.iloc[1].values, color="blue", lw=1)
ax2.set_title("AAPL", fontsize=18)
ax2.set_ylabel("Log-return", fontsize=16)
ax2.set_xlabel("Time", fontsize=16)
ax2.set_ylim(ymin, ymax)
ax2.grid(True, alpha=0.15)

# Tick labels and spines
for ax in (ax1, ax2):
    ax.tick_params(axis="both", labelsize=14, colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

plt.tight_layout()
plt.savefig("nvda_aapl_white_bigfonts.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# !pip install scikit-learn-extra
# !pip install "numpy<2.0"
# !pip install arch
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import numpy as np
from arch import arch_model

def is_strictly_stationary(alpha, beta, nu, num_samples=50000):
    rng = np.random.default_rng()
    eps = rng.standard_t(df=nu, size=num_samples)
    
    # Standardize to variance 1
    eps = eps/np.sqrt(nu / (nu - 2))
    
    vals = np.log(alpha * eps**2 + beta)
    return np.mean(vals) < 0

def compute_qaf(ts, tau_1 = 0.5, tau_2 = 0.5, lag = 1):
    ts = np.asarray(ts)
    l_ts = len(ts)

    ts_1 = ts[:l_ts - lag]
    ts_2 = ts[lag:]

    q_tau_1 = np.quantile(ts, tau_1)
    q_tau_2 = np.quantile(ts, tau_2)

    indicators_1 = (ts_1 <= q_tau_1).astype(float)
    indicators_2 = (ts_2 <= q_tau_2).astype(float)

    numerator = (1/l_ts) * np.sum(indicators_1 * indicators_2) - tau_1 * tau_2
    denominator = np.sqrt(tau_1 * (1 - tau_1) * tau_2 * (1 - tau_2))

    return numerator/denominator

def compute_qafs(ts, levels, lags):
    levels = np.asarray(levels)
    lags = np.asarray(lags)

    l_levels = len(levels)
    l_lags = len(lags)
    array_features = np.zeros((l_levels, l_levels, l_lags))

    for i in range(l_levels):
        for j in range(l_levels):
            for k in range(l_lags):
                array_features[i, j, k] = compute_qaf(
                    ts,
                    tau_1=levels[i],
                    tau_2=levels[j],
                    lag=lags[k]
                )

    return array_features.ravel(order="F")

def sample_crp_with_min_clusters(alpha, n):
    """Generate cluster assignments for n items from a CRP with concentration alpha,
    conditioned on having more than 1 cluster."""
    while True:
        assignments = []
        cluster_counts = []
        for i in range(n):
            if i == 0:
                assignments.append(0)
                cluster_counts.append(1)
            else:
                probs = np.array(cluster_counts + [alpha])
                probs = probs / probs.sum()
                choice = np.random.choice(len(probs), p=probs)
                assignments.append(choice)
                if choice == len(cluster_counts):
                    cluster_counts.append(1)
                else:
                    cluster_counts[choice] += 1
        
        assignments = np.array(assignments)
        if len(np.unique(assignments)) > 1:
            return assignments
        # else repeat the simulation until more than one cluster is produced

def simulate_garch11_clustering_with_qaf(series_length, nu_choices, random_state=None):
    rng = np.random.default_rng(random_state)

    n = 50
    alpha1 = rng.exponential(0.5)
    cluster_assignments = sample_crp_with_min_clusters(alpha1, n)

    unique_clusters = np.unique(cluster_assignments)
    K = len(unique_clusters)

    cluster_garch_params = []
    for _ in range(K):
        omega = rng.uniform(1e-6, 1e-4)
        alpha = rng.uniform(0.01, 0.30)
        beta = rng.uniform(0.70, 1.0 - alpha)
        cluster_garch_params.append((omega, alpha, beta))

    features = []

    for i in range(n):
        cluster_idx = np.where(unique_clusters == cluster_assignments[i])[0][0]
        omega, alpha, beta = cluster_garch_params[cluster_idx]

        nu = int(rng.choice(nu_choices))

        am = arch_model(None, mean="Zero", vol="GARCH", p=1, q=1, dist="t")
        sim = am.simulate([omega, alpha, beta, nu], series_length)
        ts = sim["data"].to_numpy() if hasattr(sim["data"], "to_numpy") else np.asarray(sim["data"])

        feat_vec = compute_qafs(ts, levels=(0.1, 0.5, 0.9), lags=(1, 2, 3))
        features.append(feat_vec)

    D = np.array(features)
    C = np.array(cluster_assignments)
    S = (C[:, None] == C[None, :]).astype(int)

    return D, C, S

# Example usage:
series_length = 5000
nu_choices = [3, 4, 5, 6, 7, 8, 10000]
D, C, S= simulate_garch11_clustering_with_qaf(series_length, nu_choices)
print("Feature matrix D shape:", D.shape)    
print("Cluster assignments C shape:", C.shape) 
print("Similarity matrix S shape:", S.shape)  

In [ ]:
import torch

def prepare_pairwise_data(D, S):
    ''' Returns pairs (concat) and targets for all i < j '''
    n = D.shape[0]
    pairs = []
    targets = []
    for i in range(n):
        for j in range(i+1, n):
            pair = np.concatenate([D[i], D[j]])  
            target = S[i, j]                     # 1 if same cluster, 0 if not
            pairs.append(pair)
            targets.append(target)
    return np.array(pairs), np.array(targets)

# Example preparation:
pairs, targets = prepare_pairwise_data(D, S)
print("Pairs shape:", pairs.shape)       
print("Targets shape:", targets.shape)   
print(pairs)
print(targets)

In [ ]:
# Constructing the network

import torch
import torch.nn as nn

class PairwiseNN(nn.Module):
    def __init__(self, vector_dim=27, embed_dim=128, hidden_dim=256):
        super().__init__()
        # Embedding network phi applied independently to each vector
        self.phi = nn.Sequential(
            nn.Linear(vector_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU()
        )
        # Prediction network rho that takes aggregated embedding
        self.rho = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
      
        v1 = x[:, :27]  # first vector
        v2 = x[:, 27:]  # second vector

        emb1 = self.phi(v1)  # embed first vector
        emb2 = self.phi(v2)  # embed second vector

        combined = emb1 + emb2  # symmetric aggregation (sum)

        out = self.rho(combined)  # predict
        return out.squeeze(-1)  # shape: (batch_size,)


In [ ]:
# Training

import torch
import torch.nn as nn
import numpy as np

nu_choices = [3, 4, 5, 6, 7, 8, 10000]
series_lengths = [5281]
num_epochs = 1 
units_per_epoch = 200000  # total simulated datasets per epoch
datasets_per_batch = 64   # datasets per batch update
early_stopping_patience = 100
models_by_length = {}

for T in series_lengths:
    print(f"\nTraining model for series length: {T}")
    model = PairwiseNN(vector_dim=27, embed_dim=128, hidden_dim=256)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        losses = []
        indices = np.arange(units_per_epoch // datasets_per_batch)
        for _ in indices:

            batch_num = len(losses) + 1
            print(f"Batch {batch_num}")
            
            batch_pairs = []
            batch_targets = []
            for _ in range(datasets_per_batch):
        
                D, C, S = simulate_garch11_clustering_with_qaf(series_length=T, nu_choices=nu_choices)
                pairs, targets = prepare_pairwise_data(D, S)
                batch_pairs.append(pairs)
                batch_targets.append(targets)
            
            pairs_tensor = torch.tensor(np.vstack(batch_pairs)).float()
            targets_tensor = torch.tensor(np.hstack(batch_targets)).float()
            model.train()
            optimizer.zero_grad()
            outputs = model(pairs_tensor)
            loss = criterion(outputs, targets_tensor)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = np.mean(losses)

        # Validation step
        
        model.eval()
        Dv, Cv, Sv = simulate_garch11_clustering_with_qaf(series_length=T, nu_choices=nu_choices)
        pairs_val, targets_val = prepare_pairwise_data(Dv, Sv)
        pairs_tensor_val = torch.tensor(pairs_val).float()
        targets_tensor_val = torch.tensor(targets_val).float()
        with torch.no_grad():
            outputs_val = model(pairs_tensor_val)
            val_loss = criterion(outputs_val, targets_tensor_val).item()

        print(f"Epoch {epoch+1}, train loss: {mean_loss:.4f}, val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1} (best val loss {best_val_loss:.4f})")
            break

    models_by_length[f"model_{T}"] = model

print("\nTraining completed. Models saved in memory.")

# os.makedirs("output", exist_ok=True)
# torch.save(model.state_dict(), f"output/model_{T}.pth")

In [ ]:
import numpy as np
import torch
from sklearn.cluster import SpectralClustering

# Real data matrix: rows = series (same as original)
X = df_returns.values

# Build QAF features for each real time series (same)
D_real = np.array([
    compute_qafs(ts, levels=(0.1, 0.5, 0.9), lags=(1, 2, 3))
    for ts in X
])

# Recover trained model (same)
model = models_by_length["model_5281"]

# Same helper functions for pairwise inputs and affinity prediction
def build_pairwise_inputs(D):
    n = D.shape[0]
    pairs = []
    idx = []
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append(np.concatenate([D[i], D[j]]))
            idx.append((i, j))
    return np.array(pairs), idx

def predict_affinity_matrix(model, D):
    model.eval()
    pairs, idx = build_pairwise_inputs(D)
    with torch.no_grad():
        probs = model(torch.tensor(pairs).float()).cpu().numpy().ravel()

    n = D.shape[0]
    A = np.zeros((n, n), dtype=float)
    np.fill_diagonal(A, 1.0)

    for (i, j), p in zip(idx, probs):
        A[i, j] = p
        A[j, i] = p

    return A

# New function: Spectral clustering with fixed K
def cluster_with_spectral(model, D, K, seed = 1):  # Set your desired K here
    A = predict_affinity_matrix(model, D)
    
    # SpectralClustering with precomputed affinity, fixed n_clusters=K
    spectral = SpectralClustering(
        n_clusters=K,
        affinity='precomputed',
        assign_labels='kmeans',  # Final step after eigendecomposition
        random_state=seed,
        n_init=50  # Multiple K-means runs for stability
    )
    
    labels = spectral.fit_predict(A)
    return labels, A


labels, A = cluster_with_spectral(model = model, D = D_real, K = 3)
print(labels)

In [ ]:
#### import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# =========================
# INPUT: your QAF features
# =========================
# D_real = np.array([
#     compute_qafs(ts, levels=(0.1, 0.5, 0.9), lags=(1, 2, 3))
#     for ts in X
# ])

K_min = 2  # Start from K=2
K_max = 8  # Perfect for n≈50
random_state = 4

# Create output folder
os.makedirs("output", exist_ok=True)

n_samples = D_real.shape[0]
K_max_eff = min(K_max, n_samples - 1)
K_range = list(range(K_min, K_max_eff + 1))

# Elbow computation (WCSS only)
inertias = []
print(" K |   WCSS   ")
print("-------------------------")

for K in K_range:
    kmeans = KMeans(n_clusters=K, random_state=random_state, n_init=50)
    kmeans.fit(D_real)
    
    inertias.append(kmeans.inertia_)
    print(f"{K:2d} | {kmeans.inertia_:15.2f}")

# Elbow heuristic (needs at least 3 points)
if len(inertias) >= 3:
    diff1 = np.diff(inertias)
    diff2 = np.diff(diff1)
    elbow_idx = np.argmax(diff2) + 1
    elbow_k = K_range[elbow_idx]
else:
    elbow_k = K_range[0]

# PLOT - Elbow method only, large fonts, NO bold
plt.style.use('default')
fig, ax = plt.subplots(figsize=(12, 6))

# Plot WCSS with large markers/line
ax.plot(K_range, inertias, marker='o', linewidth=4, markersize=14, 
        color='tab:blue', label='', zorder=5)

# Elbow line
ax.axvline(elbow_k, color='tab:blue', linewidth=3, linestyle='--', alpha=0.8)

# Styling with large fonts, normal weight (no bold)
ax.set_xlabel("K", fontsize=16)
ax.set_ylabel("WCSS", fontsize=16)
ax.set_title("Elbow method", 
             fontsize=20, pad=20)

ax.grid(True, alpha=0.3, linewidth=1)

# Larger ticks and labels
ax.tick_params(axis='both', labelsize=14, width=2)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.savefig("output/qaf_elbow.png", dpi=300, bbox_inches="tight", facecolor='white')
plt.show()

print(f"\nOptimal K from elbow method: {elbow_k}")
print("Plot saved: output/qaf_elbow.png")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Ensure labels are present
# labels, A = cluster_with_spectral(model=model, D=D_real, K=3)

os.makedirs("output", exist_ok=True)

# t-SNE for the full set (using raw D_real as they are already -1 to 1)
n_samples = D_real.shape[0]
tsne = TSNE(n_components=2, perplexity=min(20, n_samples - 1), random_state=2)
embed_2d = tsne.fit_transform(D_real)

# Colors: blue, orange, green
palette = ['#1f77b4', '#ff8c00', '#2ca02c']

# Cluster names
cluster_names = [f"Cluster {int(lab)+1}" for lab in labels]
# Define order to ensure 1, 2, 3
hue_order = ["Cluster 1", "Cluster 2", "Cluster 3"]

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x=embed_2d[:, 0],
    y=embed_2d[:, 1],
    hue=cluster_names,
    hue_order=hue_order,
    palette=palette,
    s=150,
    alpha=0.9
)

# Larger fonts (no bold)
plt.title('t-SNE', fontsize=24, pad=25)
plt.xlabel('Dimension 1', fontsize=20)
plt.ylabel('Dimension 2', fontsize=20)

# Legend adjustment: larger fonts and markerscale
plt.legend(fontsize=16, title_fontsize=18, frameon=True, markerscale=1.5)

plt.tick_params(axis='both', labelsize=18)

plt.tight_layout()
plt.savefig('output/qaf_tsne_all_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print("Plot saved: output/qaf_tsne_all_clusters.png")